<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

Э
В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [1]:
!pip install datasets

Импорт необходимых библиотек

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [19]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [6]:
# Загрузим датасет
dataset = load_dataset('imdb')

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: b753bced-a4c4-47e5-bfa7-b4318bdc093c)')' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/dataset_infos.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 85dd0bfc-8b2c-4841-9d6a-e571605542ea)')' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/plain_text/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
#type(dataset)
#list(dataset.keys())
dataset['train']

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [29]:
sentences = []
word_threshold = 32

for text in tqdm(dataset['train']['text']):
  text_sentences = sent_tokenize(text, language='english')

  for sent in text_sentences:
    sent_lower = sent.lower()
    words_in_sent = word_tokenize(sent_lower)

    if len(words_in_sent) < word_threshold:
      sentences.append(sent_lower)

# Получить отдельные предложения и поместить их в sentences

  0%|          | 0/25000 [00:00<?, ?it/s]

In [30]:
print("Всего предложений:", len(sentences))

Всего предложений: 198801


Посчитаем для каждого слова его встречаемость.

In [32]:
words = Counter()

for sentence in tqdm(sentences):
  tokens = word_tokenize(sentence)
  words.update(tokens)

# Расчет встречаемости слов

  0%|          | 0/198801 [00:00<?, ?it/s]

In [33]:
len(words)

68415

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [34]:
vocab = set(['<unk>', '<bos>', '<eos>', '<pad>'])
vocab_size = 40000

most_common_words = [word for word, count in words.most_common(vocab_size)]
vocab.update(most_common_words)

assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4


In [35]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [36]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [49]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
      words = word_tokenize(self.data[idx])
      tokenized_sentence = [self.bos_id]
      for word in words:
        tokenized_sentence.append(word2ind.get(word, self.unk_id))
        tokenized_sentence.append(self.eos_id)
        # Допишите код здесь

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [50]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [51]:
train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2)
eval_sentences, test_sentences = train_test_split(sentences, test_size=0.5)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [52]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            logits =model(batch['input_ids']) # Посчитайте логиты предсказаний следующих слов
            loss = criterion(logits, batch['target_ids'].flatten())
            perplexity.append(torch.exp(loss).item())

    perplexity = sum(perplexity) / len(perplexity)

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [53]:
def train_model(model, train_dataloader, eval_dataloader, criterion, optimizer, num_epochs=10, scheduler = None):
  train_losses = []
  eval_perplexities = []

  for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    batch_count = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
      optimizer.zero_grad()

      logits = model(batch['input_ids'])
      loss = criterion(logits, batch['target_ids'].flatten())

      loss.backward()
      optimizer.step()

      total_loss += loss.item()
      batch_count +=1
    avg_train_loss = total_loss/batch_count
    train_losses.append(avg_train_loss)

    eval_perplexity = evaluate(model, criterion, eval_dataloader)
    eval_perplexities.append(eval_perplexity)

    if scheduler:
      scheduler.step()
    print(f"Epoch {epoch+1}: train loss = {avg_train_loss:.4f}, eval perplexity = {eval_perplexity:.4f}")

  return train_losses, eval_perplexities

### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [54]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])

        # LSTM layers
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)

        # Output layers
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        # Embedding
        x = self.embedding(input_batch)

        # LSTM
        lstm_out, _ = self.lstm(x)

        # Apply dropout
        x = self.dropout(lstm_out)

        # Output projection
        logits = self.fc(x)

        # Reshape for cross-entropy: (batch_size * seq_len, vocab_size)
        logits = logits.reshape(-1, self.vocab_size)

        return logits

# Инициализация и обучение первой модели
vocab_size = len(vocab)
model1 = LanguageModel(vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model1.parameters(), lr=0.001)

print("Первый эксперимент: Базовая LSTM модель")
train_losses1, eval_perplexities1 = train_model(
    model1, train_dataloader, eval_dataloader, criterion, optimizer, num_epochs=5
)

# Тестирование
test_perplexity1 = evaluate(model1, criterion, test_dataloader)
print(f"Test Perplexity (Model 1): {test_perplexity1:.4f}")

Первый эксперимент: Базовая LSTM модель


Epoch 1/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 1: train loss = 2.7978, eval perplexity = 13.9999


Epoch 2/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 2: train loss = 2.6320, eval perplexity = 13.8996


...

Epoch 3: train loss = 2.6170, eval perplexity = 13.9099


Epoch 4/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 4: train loss = 2.6041, eval perplexity = 14.0323


Epoch 5/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 5: train loss = 2.5916, eval perplexity = 14.2010
Test Perplexity (Model 1): 14.2367


### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [55]:
class ImprovedLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=512, num_layers=3, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])

        # GRU layers
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers,
                         dropout=dropout, batch_first=True)

        # Additional layers
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        # Embedding
        x = self.embedding(input_batch)

        # GRU
        gru_out, _ = self.gru(x)

        # LayerNorm and dropout
        x = self.layer_norm(gru_out)
        x = self.dropout(x)

        # Output projection
        logits = self.fc(x)

        # Reshape for cross-entropy
        logits = logits.reshape(-1, self.vocab_size)

        return logits

# Второй эксперимент
model2 = ImprovedLanguageModel(vocab_size).to(device)

# Используем другой оптимизатор
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=0.001, weight_decay=0.01)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=2, gamma=0.8)

print("\nВторой эксперимент: Улучшенная GRU модель")
train_losses2, eval_perplexities2 = train_model(
    model2, train_dataloader, eval_dataloader, criterion, optimizer2,
    num_epochs=5, scheduler=scheduler2
)

# Тестирование
test_perplexity2 = evaluate(model2, criterion, test_dataloader)
print(f"Test Perplexity (Model 2): {test_perplexity2:.4f}")


Второй эксперимент: Улучшенная GRU модель


Epoch 1/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 1: train loss = 2.7847, eval perplexity = 14.1482


Epoch 2/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 2: train loss = 2.6311, eval perplexity = 14.0670


...

Epoch 3: train loss = 2.6006, eval perplexity = 14.2846


Epoch 4/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 4: train loss = 2.5786, eval perplexity = 14.2881


Epoch 5/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 5: train loss = 2.5768, eval perplexity = 14.3614
Test Perplexity (Model 2): 14.4007


третий эксперимент

In [56]:
class DeeperLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=512, num_layers=4, dropout=0.4):
        super().__init__()
        self.vocab_size = vocab_size

        # Larger embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])

        # Deep LSTM with dropout
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True, bidirectional=False)

        # Additional linear layers with residual connections
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, vocab_size)

        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.layer_norm2 = nn.LayerNorm(hidden_dim // 2)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        # Embedding
        x = self.embedding(input_batch)

        # LSTM
        lstm_out, _ = self.lstm(x)

        # First layer norm and dropout
        x = self.layer_norm1(lstm_out)
        x = self.dropout(x)

        # First linear layer with activation
        x = self.fc1(x)
        x = self.layer_norm2(x)
        x = self.activation(x)
        x = self.dropout(x)

        # Output projection
        logits = self.fc2(x)

        # Reshape for cross-entropy
        logits = logits.reshape(-1, self.vocab_size)

        return logits

# Третий эксперимент - более глубокая модель
model3 = DeeperLanguageModel(vocab_size).to(device)

# Попробуем другой learning rate и оптимизатор
optimizer3 = torch.optim.Adam(model3.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=5)

print("\nТретий эксперимент: Глубокая LSTM с дополнительными слоями")
train_losses3, eval_perplexities3 = train_model(
    model3, train_dataloader, eval_dataloader, criterion, optimizer3,
    num_epochs=5, scheduler=scheduler3
)

test_perplexity3 = evaluate(model3, criterion, test_dataloader)
print(f"Test Perplexity (Model 3): {test_perplexity3:.4f}")


Третий эксперимент: Глубокая LSTM с дополнительными слоями


Epoch 1/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 1: train loss = 2.9137, eval perplexity = 14.6018


Epoch 2/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 2: train loss = 2.7069, eval perplexity = 14.3216


...

Epoch 3: train loss = 2.6850, eval perplexity = 14.1832


Epoch 4/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 4: train loss = 2.6699, eval perplexity = 14.1017


Epoch 5/5:   0%|          | 0/1243 [00:00<?, ?it/s]

Epoch 5: train loss = 2.6591, eval perplexity = 14.0612
Test Perplexity (Model 3): 14.0801


### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

Проведено три эксперимента с языковыми моделями на основе LSTM и GRU. В первом эксперименте использовалась базовая LSTM архитектура с двумя слоями и размером скрытого состояния 256, что дало перплексию 14.24 на тестовых данных. Модель стабильно обучалась, но достигла своего предела для данной архитектуры.

Во втором эксперименте тестировалась GRU архитектура с тремя слоями, LayerNorm и увеличенным размером эмбеддингов. Несмотря на более сложную структуру, результат оказался немного хуже - перплексия 14.40. Это может быть связано с тем, что GRU хуже捕捉ляет длинные зависимости в текстах или требует больше времени на обучение.

Третий эксперимент показал наилучший результат - перплексию 14.08. Здесь использовалась глубокая LSTM архитектура с четырьмя слоями, дополнительными линейными слоями с GELU активацией и увеличенным dropout до 0.4. Также применялся более низкий learning rate с косинусным расписанием. Это подтверждает, что увеличение глубины сети вместе с правильной регуляризацией дает положительный эффект.

Для дальнейшего улучшения качества моделей можно предложить несколько направлений. Во-первых, стоит попробовать Transformer архитектуры вместо RNN, так как они лучше справляются с длинными последовательностями. Во-вторых, можно улучшить токенизацию, перейдя на subword методы типа BPE. Также полезным может быть добавление механизмов внимания к LSTM, использование bidirectional архитектур и увеличение размера эмбеддингов. Из методов обучения стоит обратить внимание на gradient clipping, learning rate warmup и early stopping. Эксперименты с разными оптимизаторами и размерами батчей также могут дать прирост качества.